# Training a chess move & board-location tagger with LFM2.5-Encoder-230M

**Goal:** give the model a piece of text, for example

> Stockfish prefers Black's **Qxd5** over **Nb8**. The game continued **Nb8 Nf3 c6 dxc6**. This puts pressure on the pawn on **d5** and opens the **c-file**.

and have it find every chess move and every board location (square, file, rank, diagonal), and tell which kind each one is. The chess-tutor app can then make all of them clickable:

| type | meaning | clicking it… |
|---|---|---|
| `START` | a move played from the current position | shows the board after: current position → this move |
| `CONT` | continues the previous move (part of a series) | shows the board after: current position → the series' START move → … → this move |
| `SQUARE` | a square: `d5` in "the pawn on d5" | highlights that square |
| `FILE` | a file: `c-file` in "the c-file is open" | highlights the whole column |
| `RANK` | a rank: `7th rank` in "a rook on the 7th rank" | highlights the whole row |
| `DIAGONAL` | a diagonal: `a1-h8` in "the a1-h8 diagonal" | highlights the diagonal |

Vague references like "the long diagonal" or "the back rank" (whose back rank?) are *not* labelled. The app couldn't tell which line to highlight.

A single move is just a `START` with nothing after it. A series is a `START` followed by `CONT`s. In "The game continued **Nb8** Nf3 c6 **dxc6**", clicking `dxc6` replays Nb8 → Nf3 → c6 → dxc6.

---

## The big picture (read this first)

**1. This is a *token classification* problem.** We don't ask the model to *write* anything. We ask it to put a label on every piece of the input text: "nothing", "a START move", "a CONT move", "a SQUARE", "a FILE" and so on. It's the same kind of task as *Named Entity Recognition* (NER), where a model finds the names of people and places in a sentence.

**2. Why an encoder?** Chatbots like ChatGPT are *decoders*: they read text left to right and predict the next word. An *encoder* reads the whole text at once, and every word can look at the words before **and after** it (*bidirectional attention*). That's exactly what we need. Whether `e4` is a move, and what kind, depends on the context on both sides:
- "White played **e4**": a move
- "the pawn on e4 is weak": a square (`SQUARE`), *not* a move
- "the c-file is open" is a `FILE`, but "upload the PGN file" is nothing
- "The game continued Nb8 Nf3 **e4**": a move that continues a series (`CONT`)

For every token, the encoder outputs a list of 1024 numbers (a *vector*) that describes that token *in its context*. By itself it doesn't output labels.

**3. Pretrained body + new head.** Liquid AI pretrained `LFM2.5-Encoder-230M` on huge amounts of text with *masked language modeling*: hide random words, make the model guess them. That's how it learned language in general. We keep that pretrained **body** (230 million parameters) and put a tiny new layer on top: a **classification head**. The head is a single linear layer that turns each token's 1024 numbers into 13 scores, one per label. Then we **fine-tune**: train the new head *and* gently adjust the whole body using our ~1000 labelled examples.

```
text ──► tokenizer ──► token ids ──► ENCODER BODY (230M params, pretrained)
                                           │
                                           ▼  1024 numbers per token
                                     HEAD (new: 1024 × 13)
                                           │
                                           ▼  13 scores per token: O, B-START, I-START, B-CONT, I-CONT, B-SQUARE, …, I-DIAGONAL
                     merge tokens back into spans ──► [("Nb8", START), ("Nf3", CONT), ("d5", SQUARE), ("c-file", FILE), …]
```

**4. What does "training" actually do?** It repeats one small cycle hundreds of times:
1. show the model a *batch* of examples,
2. it predicts a label for every token,
3. we measure how wrong it was: the **loss** (one number, lower = better),
4. **backpropagation** works out, for each of the 230M parameters, which direction would have made the loss smaller,
5. the **optimizer** nudges every parameter a tiny step in that direction.

One pass over the whole training set is called an **epoch**.

## Steps in this notebook
1. Setup · 2. Data · 3. Tokenization · 4. Labels · 5. Model · 6. Batching · 7. Measuring success · 8. Training · 9. Final test · 10. Using the model · 11. Saving

## 1. Setup

In Colab: **Runtime → Change runtime type → T4 GPU** (the free tier is enough). Then run the cells from top to bottom with **Shift+Enter**.

We need `transformers` 5.1 or newer. The encoder's custom code builds on the LFM2 model that ships inside `transformers`.

In [ ]:
%pip install -q -U "transformers>=5.1" safetensors

In [ ]:
import contextlib
import json
import os
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_model, save_model
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer, get_linear_schedule_with_warmup

MODEL_ID = "LiquidAI/LFM2.5-Encoder-230M"
SPAN_TYPES = ["START", "CONT", "SQUARE", "FILE", "RANK", "DIAGONAL"]
LABELS = ["O"] + [f"{prefix}-{kind}" for kind in SPAN_TYPES for prefix in "BI"]  # 13 labels
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))

# Hyperparameters, explained in section 8.
EPOCHS = 8
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.1
MAX_LENGTH = 512  # tokens; longer texts get cut off

DATA_DIR = Path("data")
OUTPUT_DIR = Path("chess-move-tagger")
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"  # tiny CPU run to check the notebook works; ignore in Colab

# Seeds make the "random" parts (shuffling, the head's starting weights) repeatable.
random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
if device.type == "cuda":
    # Newer GPUs (A100, L4...) support bfloat16; the T4 needs float16. See section 8.
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    print("GPU:", torch.cuda.get_device_name(), "| mixed precision:", AMP_DTYPE)
else:
    AMP_DTYPE = None
    print("No CUDA GPU found, using", device, "(training will be slow; in Colab pick a T4 runtime)")


def autocast():
    # Mixed precision only on CUDA GPUs; a no-op elsewhere.
    if AMP_DTYPE is None:
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)

## 2. The data

On your computer, run `python data/generate_dataset.py` (the files are also committed in the repo). Then upload the three files with the file picker that appears when you run the next cell:

| file | examples | what it's for |
|---|---|---|
| `train.jsonl` | 1000 | the model **learns** from these |
| `val.jsonl` | 150 | *validation*: same templates as train, but never trained on. We check them after every epoch to pick the best version of the model |
| `test.jsonl` | ~175 | the **final exam**: sentences from *held-out templates* that train never uses, plus hand-written examples (including your chess-tutor screenshots). If val is great but test is bad, the model memorised our sentence patterns instead of learning what a move is |

Each line is one JSON object:

```json
{"text": "The game continued Nb8 Nf3 c6, hitting d5.",
 "spans": [[19, 22, "START"], [23, 26, "CONT"], [27, 29, "CONT"], [39, 41, "SQUARE"]],
 "marked": "The game continued [[Nb8]] [[>Nf3]] [[>c6]], hitting [[@d5]]."}
```

`spans` are **character positions**: `text[19:22] == "Nb8"`. We store positions rather than just the move strings because the same string can be a move in one place and a square in another (`e4`), and because the app needs positions to make the text clickable. `marked` is the same thing in a human-readable form. Each type has its own prefix inside the brackets:

| markup | type | mnemonic |
|---|---|---|
| `[[Qxd5]]` | START | |
| `[[>Ne2]]` | CONT | `>` "continues" |
| `[[@e4]]` | SQUARE | `@` "at" |
| `[[\|c-file]]` | FILE | `\|` a vertical line |
| `[[_7th rank]]` | RANK | `_` a horizontal line |
| `[[/a1-h8]]` | DIAGONAL | `/` a diagonal line |

In [ ]:
FILES = ["train.jsonl", "val.jsonl", "test.jsonl"]

if not all((DATA_DIR / name).exists() for name in FILES):
    from google.colab import files  # only exists inside Colab

    DATA_DIR.mkdir(exist_ok=True)
    print("Select train.jsonl, val.jsonl and test.jsonl from the repo's data/ folder:")
    for name, content in files.upload().items():
        name = re.sub(r" \(\d+\)", "", name)  # Colab renames re-uploads to "train (1).jsonl"
        (DATA_DIR / name).write_bytes(content)


def load_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines()]


train_data, val_data, test_data = (load_jsonl(DATA_DIR / name) for name in FILES)
if SMOKE_TEST:
    train_data, val_data, test_data = train_data[:24], val_data[:8], test_data[:8]
    EPOCHS, BATCH_SIZE = 1, 4

print(f"train: {len(train_data)}  val: {len(val_data)}  test: {len(test_data)}")
for example in random.sample(train_data, 3):
    print("\n" + example["marked"])

## 3. Tokenization: how the model sees text

Models don't read characters. They read **tokens**: chunks of text from a fixed vocabulary (65,536 entries for this model), each with an integer id. Common words are a single token (` prefers`). Rare strings like chess moves get split into several pieces. Let's look:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"

sample = "Stockfish prefers Black's Qxd5 over Nb8. The game continued Nb8 Nf3 c6."
encoded = tokenizer(sample, return_offsets_mapping=True)
print(f"{'id':>6}  {'token':<18} characters")
for token_id, (start, end) in zip(encoded["input_ids"], encoded["offset_mapping"]):
    print(f"{token_id:>6}  {tokenizer.decode([token_id])!r:<18} {start:>3}–{end:<3} {sample[start:end]!r}")

Things to notice:
- **One move = several tokens.** `Qxd5` becomes ` Q`, `x`, `d`, `5`. So the model has to learn that these tokens belong together.
- **Tokens often include the space before them** (` Q`, ` over`).
- `<|startoftext|>` is a **special token** that the tokenizer adds automatically. It isn't part of the text, so its characters are `0–0` (empty).
- The **offset mapping** (the `start–end` column) says which characters each token came from. It's the bridge between our character spans and the model's tokens, in both directions.

## 4. Labels: one label per token (BIO tagging)

The model predicts **one label per token**, so we convert our character spans into token labels. We use the standard **BIO** scheme, combined with our six span types:

| label | meaning |
|---|---|
| `B-START` | **B**eginning (first token) of a START move |
| `I-START` | **I**nside a START move (its other tokens) |
| `B-CONT` / `I-CONT` | the same, for CONT moves |
| `B-SQUARE` / `I-SQUARE` | the same, for squares (` f` + `7`: squares are split into two tokens too) |
| `B-FILE` … `I-DIAGONAL` | the same, for files, ranks and diagonals |
| `O` | **O**utside: not part of any move or square |

Why the B/I split? Without it, two moves next to each other (`Nb8,Nf3` with no space) would look like one long move. `B` marks where a new move begins.

Special tokens get the label **`-100`**. PyTorch's loss function treats `-100` as "ignore this position": we don't want to train the model to classify `<|startoftext|>`.

In [ ]:
def token_labels(text, offsets, spans):
    labels = []
    for start, end in offsets:
        if start == end:  # special token: covers no characters
            labels.append(-100)
            continue
        while start < end and text[start].isspace():  # " Q" -> "Q"
            start += 1
        label = "O"
        for span_start, span_end, kind in spans:
            if start < span_end and end > span_start:  # this token overlaps this move
                label = ("B-" if start <= span_start else "I-") + kind
                break
        labels.append(LABEL2ID[label])
    return labels


def encode(example):
    enc = tokenizer(example["text"], return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
    return {"input_ids": enc["input_ids"],
            "labels": token_labels(example["text"], enc["offset_mapping"], example["spans"])}


train_features = [encode(example) for example in train_data]
val_features = [encode(example) for example in val_data]

# Look at one example token by token.
example = next(e for e in train_data if any(kind == "CONT" for *_, kind in e["spans"]))
print(example["marked"], "\n")
for token_id, label in zip(*encode(example).values()):
    print(f"{tokenizer.decode([token_id])!r:<18} {ID2LABEL.get(label, '-100 (ignored)')}")

## 5. The model: pretrained encoder + new classification head

`transformers` often gives you a ready-made `AutoModelForTokenClassification`. This model doesn't register one, so we build the head ourselves. It's only a few lines, and it shows exactly what's going on.

**Loading the body: a trap we avoid.** The model card says `AutoModel.from_pretrained(...)` loads the encoder body. With current `transformers` that **silently gives you random weights**: the checkpoint stores its weights under names like `lfm2.layers.0…`, but the bare body looks for `layers.0…`, finds nothing, and randomly initialises everything (it only prints a warning table). Instead we load the full masked-LM model, whose names match, and take its `.lfm2` attribute, which is the body. (The masked-LM's output layer shares its weights with the input embeddings, so dropping it frees almost nothing and loses nothing.)

`trust_remote_code=True` is needed because Liquid AI ships a small Python file that switches LFM2 from left-to-right to bidirectional attention. It runs code downloaded from the Hub, so only use it with repositories you trust.

The pieces of our model:
- **`nn.Dropout(0.1)`** randomly zeroes 10% of the numbers during training. This *regularises* the model: the head can't rely on one single feature, which reduces overfitting. It switches off automatically in evaluation mode (`model.eval()`).
- **`nn.Linear(1024, 13)`** is the head: 1024 × 13 weights + 13 biases = 13,325 new parameters that start out random.
- **The loss is cross-entropy.** For each token, softmax turns the 13 scores into probabilities, and the loss is `-log(probability given to the correct label)`. Confident and right → loss ≈ 0. Confident and wrong → a large loss.

In [ ]:
class ChessMoveTagger(nn.Module):
    def __init__(self, encoder, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        logits = self.classifier(self.dropout(hidden))  # shape: (batch, tokens, num_labels)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
        return loss, logits


def build_tagger():
    masked_lm = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True)
    return ChessMoveTagger(masked_lm.lfm2, num_labels=len(LABELS))


model = build_tagger().to(device)
n_body = sum(p.numel() for p in model.encoder.parameters())
n_head = sum(p.numel() for p in model.classifier.parameters())
print(f"body: {n_body / 1e6:.1f}M parameters (pretrained) | head: {n_head:,} parameters (new, random)")

## 6. Batching and padding

GPUs are fast when they process many examples at once (a **batch**). But a batch has to be a rectangle of numbers, and our texts have different lengths. So we **pad** the short ones with a `<|pad|>` token up to the longest one in the batch, and:
- the **`attention_mask`** is 1 for real tokens and 0 for padding, so the model ignores the padding,
- the **labels** for padding are `-100`, so the loss ignores it too.

The `DataLoader` shuffles the training examples every epoch, so the model doesn't see them in the same order each time.

In [ ]:
def collate(batch):
    width = max(len(item["input_ids"]) for item in batch)
    pad = lambda values, filler: values + [filler] * (width - len(values))
    return {
        "input_ids": torch.tensor([pad(item["input_ids"], tokenizer.pad_token_id) for item in batch]),
        "attention_mask": torch.tensor([pad([1] * len(item["input_ids"]), 0) for item in batch]),
        "labels": torch.tensor([pad(item["labels"], -100) for item in batch]),
    }


train_loader = DataLoader(train_features, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
first_batch = next(iter(train_loader))
print({name: tuple(tensor.shape) for name, tensor in first_batch.items()})

## 7. How we measure success

Accuracy per token would be misleading. Most tokens are `O`, so a model that never finds anything would still look "80% accurate". Instead we score **whole spans** (moves, squares, files…), which is what the app cares about:

- **Precision**: of the spans the model found, how many are real? (Low precision → it makes evaluations or `Room B12` clickable by mistake.)
- **Recall**: of the real spans, how many did it find? (Low recall → some moves or squares won't be clickable.)
- **F1**: one number combining both (their harmonic mean). 1.0 is perfect.

A prediction only counts if its start **and** end match exactly: finding `Qxd2` instead of `Qxd2+` is wrong. We report:
- **F1**: found the span **and** got its type right. This is the main number,
- **detection F1**: found the span, ignoring its type,
- **F1 per type** (`f1_START`, `f1_CONT`, `f1_SQUARE`, `f1_FILE`, `f1_RANK`, `f1_DIAGONAL`): shows *which* kind the model struggles with. For example, a low `f1_SQUARE` with a high `detection F1` means it finds `e4` but mixes up "square" and "move". Rare types (ranks, diagonals) have few examples, so their scores jump around more.

`decode` does the reverse of section 4. It turns one label per token back into character spans: consecutive tokens of the same span are merged, leading spaces are skipped, and stray punctuation (`.`, `,`, `!?`, `**`) is trimmed off the ends. `+` and `#` are kept, because they're part of the move.

In [ ]:
TRIM = " \n\t.,;:!?*`()\"'"  # never part of a move or location (but + # = - are, so they are not here)


def decode(text, offsets, label_ids):
    spans, inside_span = [], False
    for (start, end), label_id in zip(offsets, label_ids):
        while start < end and text[start].isspace():
            start += 1
        label = ID2LABEL[label_id]
        if start == end or label == "O":  # special/padding/space token, or nothing
            inside_span = False
            continue
        tag, kind = label.split("-")
        if tag == "B" or not inside_span:
            spans.append([start, end, kind])
        else:
            spans[-1][1] = end  # extend the current span
        inside_span = True
    cleaned = []
    for start, end, kind in spans:
        while start < end and text[start] in TRIM:
            start += 1
        while end > start and text[end - 1] in TRIM:
            end -= 1
        if start < end:
            cleaned.append([start, end, kind])
    return cleaned


@torch.no_grad()
def predict_spans(model, texts, batch_size=32):
    model.eval()
    results = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        enc = tokenizer(chunk, return_offsets_mapping=True, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
        with autocast():
            _, logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        predictions = logits.argmax(-1).tolist()  # the highest-scoring label for every token
        for text, offsets, label_ids in zip(chunk, enc["offset_mapping"].tolist(), predictions):
            results.append(decode(text, offsets, label_ids))
    return results


def f1_scores(gold, predicted):
    true_positives = sum(len(set(g) & set(p)) for g, p in zip(gold, predicted))
    n_predicted = sum(len(p) for p in predicted)
    n_gold = sum(len(g) for g in gold)
    precision = true_positives / n_predicted if n_predicted else 0.0
    recall = true_positives / n_gold if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def evaluate(model, examples):
    predicted = predict_spans(model, [e["text"] for e in examples])
    gold_typed = [[tuple(s) for s in e["spans"]] for e in examples]
    pred_typed = [[tuple(s) for s in p] for p in predicted]
    precision, recall, f1 = f1_scores(gold_typed, pred_typed)
    *_, detection_f1 = f1_scores([[s[:2] for s in g] for g in gold_typed], [[s[:2] for s in p] for p in pred_typed])
    metrics = {"precision": precision, "recall": recall, "f1": f1, "detection_f1": detection_f1}
    for kind in SPAN_TYPES:
        only = lambda spans: [[s for s in example if s[2] == kind] for example in spans]
        metrics[f"f1_{kind}"] = f1_scores(only(gold_typed), only(pred_typed))[2]
    return metrics, predicted


baseline, _ = evaluate(model, val_data)
print("Before training (random head):", {k: round(v, 3) for k, v in baseline.items()})

That's our baseline. With a random head the model finds (almost) nothing useful. Now let's train it.

## 8. Training

The settings (*hyperparameters*), based on the ranges the model card recommends:

| setting | value | what it means |
|---|---|---|
| epochs | 8 | passes over the 1000 training examples |
| batch size | 16 | examples per step → 63 steps per epoch, ~500 in total |
| learning rate | 5e-5 | how big each nudge is. Too high: training becomes unstable and can wreck what the body learned in pretraining. Too low: it learns very slowly. Fine-tuning uses small values because we only want to *adjust* the body, not overwrite it |
| warmup | 10% | the learning rate starts at 0, ramps up over the first 10% of steps, then decays linearly back to 0. Big early steps driven by a random head could damage the pretrained body |
| weight decay | 0.1 | gently pulls weights toward zero; another regulariser against overfitting |
| gradient clipping | 1.0 | caps a single unusually large update |
| mixed precision | bf16 / fp16 | the GPU does most of the math in 16-bit numbers (faster, less memory) while the "master" copy of the weights stays 32-bit. On a T4 we use fp16 plus a *GradScaler*, which scales the loss up so tiny gradients don't round down to zero |

**AdamW** is the optimizer: a smarter version of "step downhill" that adapts the step size for each parameter. After every epoch we score the model on `val` and **save it whenever val F1 improves**. That way we keep the best version even if later epochs start to overfit (a simple form of *early stopping*).

What to watch: the training loss should drop quickly during the first epoch, and val F1 should climb toward 1.0. Expect a few minutes on a T4.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)


def save_tagger(model, path):
    path.mkdir(parents=True, exist_ok=True)
    save_model(model, str(path / "model.safetensors"))
    tokenizer.save_pretrained(path)
    (path / "tagger_config.json").write_text(json.dumps(
        {"base_model": MODEL_ID, "labels": LABELS, "max_length": MAX_LENGTH}, indent=2))


losses, history, best_f1 = [], [], -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()  # turns dropout on
    for batch in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        batch = {name: tensor.to(device) for name, tensor in batch.items()}
        with autocast():
            loss, _ = model(**batch)                              # 1-3: predict and measure the error
        scaler.scale(loss).backward()                             # 4: backpropagation
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)                                    # 5: nudge the weights
        scaler.update()
        scheduler.step()                                          # adjust the learning rate
        optimizer.zero_grad()
        losses.append(loss.item())

    metrics, _ = evaluate(model, val_data)
    history.append(metrics)
    improved = metrics["f1"] > best_f1
    if improved:
        best_f1 = metrics["f1"]
        save_tagger(model, OUTPUT_DIR)
    print(f"epoch {epoch}: last-100-step loss {sum(losses[-100:]) / len(losses[-100:]):.4f} | "
          f"val F1 {metrics['f1']:.3f} (detection {metrics['detection_f1']:.3f})"
          + ("  ✓ saved" if improved else ""))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.5))
left.plot(losses)
left.set(title="Training loss (every step)", xlabel="step", ylabel="loss", yscale="log")
right.plot(range(1, len(history) + 1), [h["f1"] for h in history], marker="o", label="F1 (move + type)")
right.plot(range(1, len(history) + 1), [h["detection_f1"] for h in history], marker="o", label="detection F1")
right.set(title="Validation", xlabel="epoch", ylim=(0, 1.02))
right.legend()
plt.show()

## 9. The final exam: the test set

Now we **reload the best checkpoint from disk** (which also proves that saving and loading works) and score it **once** on the test set. The test set was never used to make any decision. That's why its score is an honest estimate of how the model will do on new text.

Then we print every mistake. **Mistakes are the most useful output of this notebook**: they show exactly what kind of examples to add to the data next.

In [ ]:
def load_tagger(path):
    tagger = build_tagger()  # downloads the base architecture (cached)…
    load_model(tagger, str(path / "model.safetensors"))  # …then overwrites every weight with ours
    return tagger.to(device).eval()


del model, optimizer, scaler, scheduler
if device.type == "cuda":
    torch.cuda.empty_cache()
model = load_tagger(OUTPUT_DIR)


MARKUP_PREFIX = {"START": "", "CONT": ">", "SQUARE": "@", "FILE": "|", "RANK": "_", "DIAGONAL": "/"}


def to_markup(text, spans):
    out, cursor = [], 0
    for start, end, kind in spans:
        out += [text[cursor:start], "[[", MARKUP_PREFIX[kind], text[start:end], "]]"]
        cursor = end
    return "".join(out + [text[cursor:]])


for name, subset in [("held-out templates", [e for e in test_data if e["source"] != "handwritten"]),
                     ("hand-written", [e for e in test_data if e["source"] == "handwritten"])]:
    if not subset:
        continue
    metrics, predicted = evaluate(model, subset)
    print(f"\n=== TEST · {name} ({len(subset)} examples): " + " | ".join(f"{k} {v:.3f}" for k, v in metrics.items()))
    for example, spans in zip(subset, predicted):
        if [tuple(s) for s in spans] != [tuple(s) for s in example["spans"]]:
            print("  expected:", example["marked"].replace("\n", "⏎"))
            print("  got:     ", to_markup(example["text"], spans).replace("\n", "⏎"), "\n")

## 10. Using the model

`tag_text(text)` returns everything chess-tutor needs, each item with its character `start`/`end`:

- **`series`**: a list of series, each a list of moves. The first move of a series is played from the current position, so clicking move *k* = play moves `0 … k` of that series from the current position. A single move is simply a series of length 1.
- **`highlights`**: every square, file, rank and diagonal mentioned, with its `type`, to highlight on the board when clicked.

Highlights in between don't break a series: in "Nb8 attacks d5, and after Nf3…", `Nf3` still continues `Nb8`.

In [ ]:
def tag_text(text):
    series, highlights = [], []
    for start, end, kind in predict_spans(model, [text])[0]:
        item = {"start": start, "end": end, "text": text[start:end]}
        if kind not in ("START", "CONT"):
            highlights.append({"type": kind, **item})
        elif kind == "START" or not series:
            series.append([item])
        else:
            series[-1].append(item)
    return {"series": series, "highlights": highlights}


examples = [
    "Stockfish prefers Black's Qxd5 over Nb8. In this short search, Nb8 loses about 1.8 pawns of evaluation. "
    "The game continued Nb8 Nf3 c6 dxc6 Nf6 Nd5 c7. Stockfish's line after Qxd5 begins Qxd5 Ne2.",
    "Nb8 was a mistake because it opens the position to active counterplay. The key continuation Stockfish chose was:\n\n**Qxd5**"
    "\n\nThis move puts pressure on Black's d5 pawn and opens lines for White to develop quickly.",
    "The pawn on e4 is weak, so White should play e5 and after dxe5 recapture with Nxe5, eyeing f7.",
    "Doubling rooks on the c-file with Rc1 is strong, and the bishop on the a1-h8 diagonal supports a rook lift to the 3rd rank.",
    "Upload the PGN file and I'll check your rank in the standings. The long diagonal is a hotel bar, by the way.",
    "The game ended 1-0. Room B12, board 4, printed on A4 paper.",
]
for text in examples:
    print(text.replace("\n", "⏎"))
    tags = tag_text(text)
    for i, moves in enumerate(tags["series"], 1):
        print(f"   series {i}: " + " → ".join(m["text"] for m in moves))
    print("   highlights:", [f"{h['text']} ({h['type']})" for h in tags["highlights"]], "\n")

## 11. Saving and downloading

The best checkpoint is already saved in `chess-move-tagger/`:

```
chess-move-tagger/
  model.safetensors     # all weights: fine-tuned body + head (~920 MB)
  tagger_config.json    # label names, base model id
  tokenizer.json, ...   # the tokenizer
```

To use it later: define `ChessMoveTagger`, `build_tagger` and `load_tagger` as above, then call `load_tagger(Path("chess-move-tagger"))`.

**Colab deletes all files when the session ends.** Copy the folder to Google Drive (fast), or download a zip (slower, ~850 MB):

In [ ]:
if not SMOKE_TEST:
    from google.colab import drive

    drive.mount("/content/drive")
    !cp -r chess-move-tagger /content/drive/MyDrive/
    print("Saved to Google Drive → MyDrive/chess-move-tagger")

    # Or download a zip instead:
    # !zip -r chess-move-tagger.zip chess-move-tagger
    # from google.colab import files; files.download("chess-move-tagger.zip")

## Where to go from here

- **Add real data.** Paste real chess-tutor answers into `data/handwritten.txt`, mark the moves and locations with the markup from section 2, and look at where the model is wrong. The examples it gets wrong are the most valuable ones to add. Once you have a few hundred real examples, train on some of them too, not only test.
- **If test F1 is much lower than val F1**, the model is overfitting to the templates: add more varied templates to `data/generate_dataset.py`.
- **Verify in the app.** Before navigating, replay the series with a chess library from the current position. If a move is illegal (the model was wrong, or the text talks about some other position), just don't make it clickable.